In [11]:
import random
import csv
from pathlib import Path
import pandas as pd

heerlen_addresses_df = pd.read_csv("../output/heerlen_addresses.csv")

# Use prevalidated addresses and coordinates, same pattern as clients generator.
required_columns = {"full_address", "coordinates"}
missing_columns = required_columns - set(heerlen_addresses_df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns in heerlen_addresses_df: {sorted(missing_columns)}")

VALID_HEERLEN_ADDRESS_ROWS = (
    heerlen_addresses_df[["full_address", "coordinates"]]
    .dropna(subset=["full_address"])
    .assign(full_address=lambda d: d["full_address"].astype(str).str.strip())
    .drop_duplicates(subset=["full_address"])
    .to_dict("records")
)

if not VALID_HEERLEN_ADDRESS_ROWS:
    raise ValueError("No usable rows found in heerlen_addresses_df['full_address'].")

random.shuffle(VALID_HEERLEN_ADDRESS_ROWS)
_ADDRESS_INDEX = 0

WORK_DAYS = ["monday", "tuesday", "wednesday", "thursday", "friday"]
WORK_DAY_ORDER = {day: idx for idx, day in enumerate(WORK_DAYS)}

DAY_START_MIN = 7 * 60   # 07:00
DAY_END_MIN = 19 * 60    # 19:00
MIN_SHIFT_MIN = 3 * 60   # 3 hours
MAX_SHIFT_MIN = 6 * 60   # 6 hours
SHIFT_STEP_MIN = 30      # 30-minute intervals

DUTCH_FIRST_NAMES = [
    "Daan", "Lars", "Sven", "Bram", "Thijs", "Sem", "Noah", "Ruben", "Milan", "Jesse",
    "Emma", "Sanne", "Lotte", "Julia", "Sophie", "Noor", "Fleur", "Lisa", "Eva", "Nina",
    "Tess", "Anne", "Mila", "Roos", "Yara", "Esmee", "Puck", "Iris", "Lauren", "Maud",
    "Niels", "Stijn", "Finn", "Levi", "Thomas", "Timo", "Wout", "Luuk", "Koen", "Joris",
    "Hidde", "Mats", "Dani", "Bo", "Romy", "Jasmijn", "Marit", "Pien", "Femke", "Elin",
]
DUTCH_LAST_NAMES = [
    "de Vries", "Jansen", "de Jong", "Bakker", "Visser", "Smit", "Meijer", "Mulder", "Bos", "Dekker",
    "Peters", "Hendriks", "van Dijk", "van den Berg", "van Leeuwen", "Kuipers", "Jacobs", "de Boer", "Vos", "Schouten",
    "Willems", "Hermans", "Kok", "Vermeulen", "de Graaf", "Martens", "van Loon", "van der Meer", "Kramer", "Postma",
    "Prins", "Hoekstra", "Smits", "Wouters", "Evers", "van Vliet", "Mol", "Groen", "Brouwer", "Koster",
    "van Beek", "Kool", "Molenaar", "Schaap", "Koning", "Veldhuis", "Hofman", "Kleijn", "Wessels", "Rood",
]


def _minutes_to_hhmm(total_minutes: int) -> str:
    hours = total_minutes // 60
    minutes = total_minutes % 60
    return f"{hours:02d}:{minutes:02d}"


def _generate_shift_window():
    """Generate one shift window and its available hours."""
    shift_duration_min = random.randrange(MIN_SHIFT_MIN, MAX_SHIFT_MIN + SHIFT_STEP_MIN, SHIFT_STEP_MIN)
    latest_start = DAY_END_MIN - shift_duration_min
    shift_start_min = random.randrange(DAY_START_MIN, latest_start + SHIFT_STEP_MIN, SHIFT_STEP_MIN)
    shift_end_min = shift_start_min + shift_duration_min

    time_window_start = _minutes_to_hhmm(shift_start_min)
    time_window_end = _minutes_to_hhmm(shift_end_min)
    available_hours = shift_duration_min / 60.0

    return time_window_start, time_window_end, available_hours


def _random_dutch_fullname():
    first_name = random.choice(DUTCH_FIRST_NAMES)
    last_name = random.choice(DUTCH_LAST_NAMES)
    return f"{first_name} {last_name}"


def generate_real_address(max_attempts: int = 1):
    """Return one known Heerlen address row with its coordinates."""
    global _ADDRESS_INDEX
    row = VALID_HEERLEN_ADDRESS_ROWS[_ADDRESS_INDEX % len(VALID_HEERLEN_ADDRESS_ROWS)]
    _ADDRESS_INDEX += 1
    return row


def generate_employee(index):
    """Generate a single employee record as a dictionary."""
    name = f"employee {index}"
    fullname = _random_dutch_fullname()

    address_row = generate_real_address()
    address = address_row["full_address"]
    coordinates = address_row.get("coordinates")

    time_window_start, time_window_end, available_hours = _generate_shift_window()

    availability_days = random.sample(WORK_DAYS, k=3)
    availability_days = sorted(availability_days, key=lambda d: WORK_DAY_ORDER[d])
    availability = ",".join(availability_days)

    dogs = random.choices([0, 1, 2, -1], weights=[0.2, 0.2, 0.1, 0.5])[0]
    cats = random.choices([0, 1, 2, -1], weights=[0.1, 0.3, 0.1, 0.5])[0]
    smokes = random.random() < 0.3

    return {
        "name": name,
        "fullname": fullname,
        "address": address,
        "coordinates": coordinates,
        "time_window_start": time_window_start,
        "time_window_end": time_window_end,
        "available_hours": available_hours,
        "availability": availability,
        "dogs": dogs,
        "cats": cats,
        "smokes": smokes,
    }


def generate_employees_csv(num_employees, filename="../output/employees.csv"):
    """Generate num_employees records and write them to a CSV file."""
    fieldnames = [
        "name", "fullname", "address", "coordinates",
        "time_window_start", "time_window_end", "available_hours", "availability",
        "dogs", "cats", "smokes",
    ]

    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, mode="w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for i in range(1, num_employees + 1):
            employee = generate_employee(i)
            employee["available_hours"] = f"{employee['available_hours']:.1f}"
            employee["smokes"] = str(employee["smokes"]).lower()
            writer.writerow(employee)

    print(f"Generated {num_employees} employees in '{output_path}'.")


if __name__ == "__main__":
    generate_employees_csv(20)

Generated 20 employees in '..\output\employees.csv'.
